# meta-agent-gym: GRPO Training (Colab T4)

Train a policy to generate AGENT.md files using GRPO with Unsloth 4-bit LoRA.

**Steps:**
1. Clean environment + install deps (fails loudly if setup breaks)
2. Clone repo
3. Collect baselines (random + heuristic)
4. Run expert benchmark (upper bound)
5. GRPO training — dry-run then real
6. Evaluate trained model (LoRA adapter rollouts)
7. Generate monitoring plots
8. Generate comparison plots (baseline vs trained)
9. Package + download results

**Policy: fail loud, no mock fallbacks.** Any dependency error, missing file, or training failure stops the notebook so you can see and fix it — rather than generating plausible-looking placeholder output.

**Key install fix:** xformers is pinned to a prebuilt wheel version and installed with `--no-deps` to avoid the source build that was failing on Colab.

In [1]:
#@title 1. Clean environment + Setup (run FIRST)
#@markdown Pins aligned to Unsloth 2026.4.8 requirements. Fails loudly if any import is broken.

import torch
import os
import sys

# Fail fast if no GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Requires a GPU runtime: "
        "Runtime -> Change runtime type -> T4 GPU (or better)."
    )
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Remove anything that could conflict
!pip uninstall -y trl unsloth unsloth_zoo pydantic xformers transformers datasets 2>/dev/null

# Pydantic (TRL <-> pydantic conflict pinned per commit 802e79e)
!pip install -q "pydantic>=2.0,<2.11"

# Core ML stack — versions aligned to Unsloth 2026.4.8 requirements
# unsloth requires: transformers !=5.0.0,<=5.5.0,>=4.51.3 and datasets !=4.0.*,<4.4.0,>=3.4.1
!pip install -q "transformers>=4.51.3,<5.0.0" "datasets>=3.4.1,<4.0.0" "accelerate>=0.24.0" "peft>=0.10.0" "bitsandbytes>=0.43.0"

# xformers: prebuilt wheel (avoids source build that was failing on Colab)
!pip install -q --no-deps "xformers==0.0.29.post3"

# TRL — Unsloth 2026.4.8 requires >=0.18.2,<=0.24.0 (excludes 0.19.0)
!pip install -q "trl>=0.18.2,!=0.19.0,<=0.24.0"

# Unsloth — --no-deps prevents it from dragging xformers-from-source
!pip install -q --no-deps "unsloth_zoo" "hf_transfer" "tyro"
!pip install -q --no-deps "unsloth"

# Remaining deps
!pip install -q "httpx>=0.24.0" "matplotlib>=3.7.0" "pandas>=1.5.0" "numpy>=1.24.0" "pyyaml>=6.0" "sentencepiece" "protobuf"

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# --- Unsloth wants to be imported BEFORE trl/transformers/peft for full patching ---
import unsloth  # noqa: F401 — order matters for the optimization patches
print(f"[OK] Unsloth {getattr(unsloth, '__version__', 'unknown')} imported first")

import pydantic
print(f"[OK] Pydantic {pydantic.__version__}")

from trl import GRPOConfig, GRPOTrainer  # noqa: F401
print("[OK] TRL GRPOTrainer imported")

import xformers
print(f"[OK] xformers {xformers.__version__}")

import transformers
print(f"[OK] transformers {transformers.__version__}")

import datasets
print(f"[OK] datasets {datasets.__version__}")

print("\n=== Setup complete — all dependencies verified ===")

PyTorch: 2.10.0+cu128
GPU: Tesla T4
GPU Memory: 15.6 GB
Found existing installation: pydantic 2.12.3
Uninstalling pydantic-2.12.3:
  Successfully uninstalled pydantic-2.12.3
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mcp 1.27.0 requires pydantic<3.0.0,>=2.11.0, but you have pydantic 2.10.6 which is incompatible.
google-adk 1.29.0 requires pydantic<3.0.0,>=2.12.0, but you have pydantic 2.10.6 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0

In [2]:
#@title 2. Clone repo + install openenv-core only

REPO_URL = 'https://github.com/Kaviyamurugadass/meta-agent-gym.git' #@param {type:'string'}

import os, sys

print(f'[...] Cloning {REPO_URL}')
!git clone {REPO_URL} /content/openenv-agent-gym

sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

# Install openenv-core explicitly (not the full project) so models.py gets the
# proper base classes. We SKIP `pip install -e .` because it pulls in project
# deps that conflict with cell 1's carefully pinned versions (transformers
# 5.0.0, datasets 4.0.0, trl 0.16.1 etc.). The code runs fine from sys.path.
!pip install -q "openenv-core==0.2.1" "fastapi" "uvicorn[standard]" "websockets"

print('[...] Verifying imports...')
from server.environment import Environment  # noqa: F401
print('[OK] Environment imported')

from training.rollout_collection import collect  # noqa: F401
print('[OK] Rollout collection imported')

# Sanity-check: does Observation have .done? (regression guard for earlier bug)
from models import Observation
assert hasattr(Observation(task_id='x', step=0, max_steps=1), 'done'), \
    "Observation.done missing â€” models.py fix not applied. Re-pull the repo."
print('[OK] Observation.done present')

print('\n=== Project loaded and ready ===')

[...] Cloning https://github.com/Kaviyamurugadass/meta-agent-gym.git
Cloning into '/content/openenv-agent-gym'...
remote: Enumerating objects: 2898, done.
remote: Counting objects: 100% (288/288), done.
remote: Compressing objects: 100% (156/156), done.
remote: Total 2898 (delta 151), reused 248 (delta 129), pack-reused 2610 (from 4)
Receiving objects: 100% (2898/2898), 4.77 MiB | 17.50 MiB/s, done.
Resolving deltas: 100% (827/827), done.
Filtering content: 100% (32/32), 80.68 MiB | 12.61 MiB/s, done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━

In [3]:
#@title 3. Collect baselines (random + heuristic)

import sys, os
sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

from training.rollout_collection import collect

print('Collecting random baseline (20 episodes)...')
random_ds = collect(episodes=20, policy='random', output_dir='data/baseline/random', seed=42)
print(f'Random: {random_ds.summary()}')

print('\nCollecting heuristic baseline (20 episodes)...')
heuristic_ds = collect(episodes=20, policy='heuristic', output_dir='data/baseline/heuristic', seed=42)
print(f'Heuristic: {heuristic_ds.summary()}')

[1/20] task=an_easy_001 length=7 reward=0.000 success=False
[2/20] task=fi_easy_002 length=7 reward=0.000 success=False
[3/20] task=da_easy_001 length=7 reward=0.000 success=False
[4/20] task=ws_easy_001 length=7 reward=0.000 success=False
[5/20] task=cr_easy_001 length=7 reward=0.000 success=False
[6/20] task=an_easy_001 length=7 reward=0.000 success=False
[7/20] task=an_easy_001 length=7 reward=0.000 success=False
[8/20] task=fi_easy_002 length=7 reward=0.000 success=False
[9/20] task=fi_easy_001 length=7 reward=0.000 success=False
[10/20] task=fi_easy_001 length=7 reward=0.000 success=False
[11/20] task=te_easy_001 length=7 reward=0.000 success=False
[12/20] task=ws_easy_001 length=7 reward=0.000 success=False
[13/20] task=fi_easy_001 length=7 reward=0.000 success=False
[14/20] task=rf_easy_001 length=7 reward=0.000 success=False
[15/20] task=fi_easy_001 length=7 reward=0.000 success=False
[16/20] task=fi_easy_002 length=7 reward=0.000 success=False
[17/20] task=cr_easy_001 length=7

In [4]:
#@title 4. Run expert benchmark (upper bound)

import sys, os
sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

from training.benchmark import run_all

print('Running expert benchmark...')
results = run_all()
for r in results:
    print(f'{r.scenario_name:30s} reward={r.total_reward:7.3f} steps={r.steps_taken:3d} success={r.success}')

successes = [r for r in results if r.success]
if not successes:
    raise RuntimeError(
        "No expert scenarios succeeded â€” check scenarios/environment before training. "
        "Training with a broken environment wastes compute."
    )
expert_mean = sum(r.total_reward for r in successes) / len(successes)
print(f'\nExpert upper bound: mean_reward={expert_mean:.3f} ({len(successes)}/{len(results)} scenarios succeeded)')

Running expert benchmark...
placeholder_easy               reward=  0.000 steps=  4 success=False
ws_easy_001                    reward= 11.700 steps=  6 success=True
da_easy_001                    reward= 11.633 steps=  6 success=True
cr_easy_001                    reward= 11.033 steps=  6 success=True
fi_easy_001                    reward= 19.567 steps=  6 success=True
fi_easy_002                    reward= 17.967 steps=  6 success=True
an_easy_001                    reward= 15.967 steps=  6 success=True
ou_easy_001                    reward= 10.967 steps=  6 success=True
ws_medium_001                  reward= 15.467 steps=  8 success=True
da_medium_001                  reward= 15.967 steps=  7 success=True
cr_medium_001                  reward= 16.633 steps=  7 success=True
fi_medium_001                  reward= 15.967 steps=  8 success=True
an_medium_001                  reward= 15.967 steps=  7 success=True
ws_hard_001                    reward= 15.067 steps= 10 success=True
da_ha

In [5]:
#@title Debug toggles (optional) — show model outputs in Cell 5
import os

# Print the first few raw model completions from reward_fn
os.environ["SHOW_COMPLETIONS"] = "1"
os.environ["SHOW_COMPLETIONS_LIMIT"] = "4"

# Print first few completions from adapter policy (eval / Cell 7)
os.environ["SHOW_ADAPTER_COMPLETIONS"] = "1"
os.environ["SHOW_ADAPTER_COMPLETIONS_LIMIT"] = "3"

# If you want training to crash on the first reward_fn error (to see traceback), uncomment:
# os.environ["REWARD_FN_FAIL_FAST"] = "1"

In [6]:
#@title 5. GRPO Training (dry-run first, then real)

import sys, os, json
sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

MODEL_ID = 'Qwen/Qwen3-1.7B' #@param ['Qwen/Qwen3-1.7B', 'Qwen/Qwen2.5-0.5B', 'Qwen/Qwen2-0.5B']
NUM_EPOCHS = 2 #@param {type:'integer'}
NUM_GENERATIONS = 2 #@param {type:'integer'}
DATASET_EPISODES = 25 #@param {type:'integer'}

# Re-verify critical deps (cell 1 already checked, but this cell may be re-run standalone)
from trl import GRPOConfig, GRPOTrainer  # noqa: F401
import unsloth  # noqa: F401
from training.grpo_unsloth import main

base_argv = [
    'grpo_unsloth.py',
    '--model-id', MODEL_ID,
    '--num-epochs', str(NUM_EPOCHS),
    '--num-generations', str(NUM_GENERATIONS),
    '--dataset-episodes', str(DATASET_EPISODES),
    '--max-seq-length', '768',  # T4 memory constraint
    '--max-completion-length', '384',  # enough for a 6-action JSON array; thinking suppressed via chat_template
    '--per-device-train-batch-size', '1',
    '--gradient-accumulation-steps', '4',
    '--learning-rate', '5e-6',
    '--output-dir', 'training/grpo-unsloth-output',
    '--seed', '42',
]

# Step A: Dry-run — validates config and imports without burning GPU time
print('=== Step A: Dry-run validation ===')
sys.argv = base_argv + ['--dry-run']
main()
print('[OK] Dry-run passed\n')

# Step B: Real training
print('=== Step B: Real GRPO training ===')
sys.argv = base_argv  # no --dry-run
main()
print('[OK] Training complete')

# Real-training sentinel — lets downstream cells and post-hoc checks distinguish
# this run from a mock/replay. If this file is missing, assume training did NOT run.
os.makedirs('training/grpo-unsloth-output', exist_ok=True)
summary_path = 'training/grpo-unsloth-output/training_summary.json'
with open(summary_path, 'w') as f:
    json.dump({
        "real_training": True,
        "model": MODEL_ID,
        "dataset_episodes": DATASET_EPISODES,
        "num_epochs": NUM_EPOCHS,
        "num_generations": NUM_GENERATIONS,
    }, f, indent=2)
print(f'Wrote training sentinel: {summary_path}')

=== Step A: Dry-run validation ===
=== GRPO Unsloth dry-run ===
model: Qwen/Qwen3-1.7B
output: training/grpo-unsloth-output
num_generations: 2
max_seq_length: 768
4bit: True
lora: r=16 alpha=16 dropout=0.0
reward backend built: LocalBackend
unsloth available

[OK] Dry-run passed. Remove --dry-run to launch real training.
[OK] Dry-run passed

=== Step B: Real GRPO training ===
==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.41G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...


============================== MODEL COMPLETION ==============================
[completion] scenario='da_easy_001'
<think>

</think>

[
  {"command": "set_name", "args": {"name": "Agent Data"}},
  {"command": "set_description", "args": {"description": "Agent for data tasks"}},
  {"command": "add_skill", "args": {"skill": "xlsx"}},
  {"command": "write_prompt", "args": {"prompt": "Efficiently process and analyze data using Excel"}},
  {"command": "set_model", "args": {"model": "sonnet"}},
  {"command": "submit", "args": {}}
]


============================== MODEL COMPLETION ==============================
[completion] scenario='da_easy_001'
<think>

</think>

[
  {"command": "set_name", "args": {"name": "Data Analyst"}},
  {"command": "set_description", "args": {"description": "Analyzes and interprets data to support decision-making"}},
  {"command": "add_skill", "args": {"skill": "xlsx"}},
  {"command": "write_prompt", "args": {"prompt": "Identify trends, generate reports, and provide


============================== MODEL COMPLETION ==============================
[completion] scenario='db_easy_001'
<think>

</think>

[
  {"command": "set_name", "args": {"name": "CodeSolver"}},
  {"command": "set_description", "args": {"description": "A code solver that analyzes and fixes issues in software"}},
  {"command": "add_skill", "args": {"skill": "log-analyzer"}},
  {"command": "write_prompt", "args": {"prompt": "Analyze logs to identify issues and fix code errors in real-time"}},
  {"command": "set_model", "args": {"model": "sonnet"}},
  {"command": "submit", "args": {}}
]


============================== MODEL COMPLETION ==============================
[completion] scenario='db_easy_001'
<think>

</think>

[
  {"command": "set_name", "args": {"name": "CodeBot"}},
  {"command": "set_description", "args": {"description": "A code assistant for log analysis and code fixing"}},
  {"command": "add_skill", "args": {"skill": "log-analyzer"}},
  {"command": "write_prompt", "args": {


============================== MODEL COMPLETION ==============================
[completion] scenario='te_easy_001'
<think>

</think>

[
  {"command": "set_name", "args": {"name": "te_easy_001"}},
  {"command": "set_description", "args": {"description": "A simple test environment for easy tasks"}},
  {"command": "add_skill", "args": {"skill": "test-driven-development"}},
  {"command": "write_prompt", "args": {"prompt": "Write a test suite to verify the correctness of the solution"}]},
  {"command": "set_model", "args": {"model": "sonnet"}},
  {"command": "submit", "args": {}}
]


============================== MODEL COMPLETION ==============================
[completion] scenario='te_easy_001'
<think>

</think>

[
  {"command": "set_name", "args": {"name": "te_easy_001"}},
  {"command": "set_description", "args": {"description": "A simple test environment for code development"}},
  {"command": "add_skill", "args": {"skill": "test-driven-development"}},
  {"command": "write_prompt", "arg


============================== MODEL COMPLETION ==============================
[completion] scenario='da_easy_001'
<think>

</think>

[
  {"command": "set_name", "args": {"name": "Data Analyst"}},
  {"command": "set_description", "args": {"description": "Analyzes and interprets data to support decision-making"}},
  {"command": "add_skill", "args": {"skill": "xlsx"}},
  {"command": "write_prompt", "args": {"prompt": "Identify trends, generate insights, and present findings in clear reports"}},
  {"command": "set_model", "args": {"model": "sonnet"}},
  {"command": "submit", "args": {}}
]


============================== MODEL COMPLETION ==============================
[completion] scenario='da_easy_001'
<think>

</think>

[
  {"command": "set_name", "args": {"name": "Data Analyst"}},
  {"command": "set_description", "args": {"description": "Analyze and interpret data to support decision-making"}},
  {"command": "add_skill", "args": {"skill": "xlsx"}},
  {"command": "write_prompt", "args"


============================== MODEL COMPLETION ==============================
[completion] scenario='fi_easy_001'
<think>

</think>

[
  {"command": "set_name", "args": {"name": "File Reader"}},
  {"command": "set_description", "args": {"description": "An agent that reads files efficiently"}},
  {"command": "add_skill", "args": {"skill": "file-reader"}},
  {"command": "write_prompt", "args": {"prompt": "Read and analyze files to complete tasks"}},
  {"command": "set_model", "args": {"model": "sonnet"}},
  {"command": "submit"}
]


============================== MODEL COMPLETION ==============================
[completion] scenario='fi_easy_001'
<think>

</think>

[
  {"command": "set_name", "args": {"name": "File Reader"}},
  {"command": "set_description", "args": {"description": "An agent that reads and processes files efficiently"}},
  {"command": "add_skill", "args": {"skill": "file-reader"}},
  {"command": "write_prompt", "args": {"prompt": "Read and analyze files to complete task

In [7]:
#@title 6. Evaluate trained model (LoRA adapter rollouts)

import sys, os, json
sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

from training.rollout_collection import collect
from training.evaluation import EvaluationSuite
from training.trajectory import TrajectoryDataset

# Hard gate: baselines must exist from cell 3
for d in ['data/baseline/random', 'data/baseline/heuristic']:
    if not os.path.exists(d) or not os.listdir(d):
        raise RuntimeError(f"Baseline missing: {d}. Re-run cell 3.")

# Hard gate: training sentinel must exist from cell 5
sentinel = 'training/grpo-unsloth-output/training_summary.json'
if not os.path.exists(sentinel):
    raise RuntimeError(f"No training sentinel at {sentinel}. Re-run cell 5 (training did not complete).")

# Hard gate: adapter must exist
adapter_dir = 'training/grpo-unsloth-output'
if not os.path.exists(os.path.join(adapter_dir, 'adapter_config.json')):
    raise RuntimeError(
        f"No adapter_config.json in {adapter_dir}. "
        "Training did not produce a LoRA adapter — check cell 5 output."
    )

# Use the same base model you trained
with open(sentinel, 'r') as f:
    sentinel_data = json.load(f)
base_model = sentinel_data.get('model')
if not base_model:
    raise RuntimeError(f"Sentinel missing model field: {sentinel}")

print('=' * 70)
print('Collecting rollouts with policy=adapter')
print(f'  base_model  : {base_model}')
print(f'  adapter_path: {adapter_dir}')
print('=' * 70)

trained_ds = collect(
    episodes=10,
    policy='adapter',
    adapter_path=adapter_dir,
    base_model=base_model,
    output_dir='data/trained',
    seed=123,
)

random_baseline = TrajectoryDataset.load_dir('data/baseline/random')
heuristic_baseline = TrajectoryDataset.load_dir('data/baseline/heuristic')
print(f'Loaded baselines: random={len(random_baseline)} heuristic={len(heuristic_baseline)}')

report = EvaluationSuite.full_report(trained_ds, reference=heuristic_baseline, label='adapter_trained')
print('\n=== Adapter-Trained Model Report ===')
print(json.dumps(report, indent=2, default=str))

  base_model  : Qwen/Qwen3-1.7B
  adapter_path: training/grpo-unsloth-output
==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
[adapter_policy] Loaded Unsloth model + adapter
============================== ADAPTER COMPLETION #1 ==============================
[adapter] task=an_easy_001 step=0 (new episode -> generating)
[adapter] raw_response (len=437):
```json
[
  {"command": "set_name", "args": {"name": "Agent"}},
  {"command": "set_description", "args": {"description": "A self-improving agen

In [8]:
#@title 7. Generate monitoring plots

import sys, os
sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

import matplotlib  # noqa: F401
from training.monitoring import TrainingMonitor

monitor = TrainingMonitor(window=10)

# Hard gate: all three data sources must exist
required_dirs = ['data/baseline/random', 'data/baseline/heuristic', 'data/trained']
for d in required_dirs:
    if not os.path.exists(d):
        raise RuntimeError(f"Required data directory missing: {d}")
    monitor.ingest_dir(d)
    print(f'[OK] Ingested {d}')

monitor.print_summary()

os.makedirs('monitoring', exist_ok=True)
monitor.save_json('monitoring/report.json')
monitor.plot('monitoring', title='Training Progress')
print('\nMonitoring plots saved to monitoring/')

[OK] Ingested data/baseline/random
[OK] Ingested data/baseline/heuristic
[OK] Ingested data/trained

Training Monitor — 50 episodes
  Total reward:  mean=10.293  std=9.107
                 min=0.000  max=20.333
                 trend=+0.4025/ep
  Success rate:  100.0%

  Component breakdown:
  * best_practices                 mean=+0.000  trend=+0.0000
    core                           mean=+4.222  trend=-0.0685
  * description_quality            mean=+0.256  trend=+0.0097
  * efficiency                     mean=+1.000  trend=+0.0000
    gate_failed                    mean=+1.000  trend=+0.0000
    gated                          mean=+1.000  trend=+0.0000
    has_required_fields            mean=+0.329  trend=+0.0151
  * model_appropriateness          mean=+0.800  trend=-0.0000
    model_valid                    mean=+1.000  trend=+0.0000
    novelty_bonus                  mean=+0.100  trend=+0.0000
    progress                       mean=+0.267  trend=+0.0022
    prompt_length_ok     

In [9]:
#@title 8. Generate comparison plots (baseline vs trained)

import sys, os
sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

from training.plot_rewards import plot_compare

input_dirs = ['data/baseline/random', 'data/baseline/heuristic', 'data/trained']
labels = ['Random Baseline', 'Heuristic Baseline', 'GRPO Trained']

for d in input_dirs:
    if not os.path.exists(d):
        raise RuntimeError(f"Required directory missing: {d}. Run earlier cells first.")

os.makedirs('monitoring', exist_ok=True)
plot_compare(
    input_dirs=input_dirs,
    labels=labels,
    output_path='monitoring/full_comparison.png',
    title='Before/After: Baseline vs GRPO Trained',
)
print(f'[OK] Comparison plot generated with {len(input_dirs)} datasets -> monitoring/full_comparison.png')

saved: monitoring/full_comparison.png
  Random Baseline: n=20 mean=0.00 best=0.00
  Heuristic Baseline: n=20 mean=20.33 best=20.33
  GRPO Trained: n=10 mean=10.80 best=11.60
[OK] Comparison plot generated with 3 datasets -> monitoring/full_comparison.png


In [11]:
#@title 9. Download results

import sys, os, json, shutil
sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

try:
    from google.colab import files
    COLAB_AVAILABLE = True
    print('Google Colab environment detected')
except ImportError:
    COLAB_AVAILABLE = False
    print('Not in Google Colab â€” files will be listed for manual download')

os.makedirs('results', exist_ok=True)

# Package artifacts â€” must exist (earlier cells would have failed otherwise)
artifacts = [
    ('monitoring', 'results/meta-agent-gym-results'),
    ('data/trained', 'results/meta-agent-gym-trained'),
    ('training/grpo-unsloth-output', 'results/meta-agent-gym-model'),
]
created = []
for src, dest in artifacts:
    if not os.path.exists(src):
        raise RuntimeError(f"Expected artifact missing: {src}. Did earlier cells complete?")
    shutil.make_archive(dest, 'zip', src)
    zip_path = f'{dest}.zip'
    created.append(zip_path)
    print(f'[OK] {zip_path}')

# Write a summary that includes the real-training sentinel content so reviewers
# can see whether this run was real.
sentinel_path = 'training/grpo-unsloth-output/training_summary.json'
sentinel = json.load(open(sentinel_path)) if os.path.exists(sentinel_path) else {"real_training": False}

summary = {
    'notebook': 'train_colab.ipynb',
    'training_sentinel': sentinel,
    'zips': created,
}
with open('results/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('\n' + '=' * 50)
if COLAB_AVAILABLE:
    print('Downloading to your machine...')
    for path in created + ['results/summary.json']:
        if os.path.exists(path):
            print(f'[...] {path}')
            files.download(path)
else:
    print('Files ready (manual download):')
    for path in created + ['results/summary.json']:
        if os.path.exists(path):
            print(f'  {path}')

print('\n[DONE] Notebook execution complete.')

Google Colab environment detected
[OK] results/meta-agent-gym-results.zip
[OK] results/meta-agent-gym-trained.zip
[OK] results/meta-agent-gym-model.zip

[...] results/meta-agent-gym-results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[...] results/meta-agent-gym-trained.zip


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

[...] results/summary.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


[DONE] Notebook execution complete.
